# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 5: Mini-GPT - pełny transformer od zera

### Materiały i inspiracje

Ten notebook to bezpośrednia kontynuacja [Notebooka 4](4_self_attention.ipynb) i opiera się na tych samych źródłach:

- [Let's build GPT: from scratch, in code, spelled out (YouTube)](https://www.youtube.com/watch?v=kCc8FmEb1nY) - Andrej Karpathy. Architektura mini-GPT w tym notebooku jest świadomie blisko jego implementacji z wykładu.
- [nanoGPT (GitHub)](https://github.com/karpathy/nanoGPT) - referencyjny minimalny GPT, ~300 linii kodu.
- [Attention Is All You Need (Vaswani i in., 2017)](https://arxiv.org/abs/1706.03762) - oryginalny transformer.
- [GPT-2 (Radford i in., 2019)](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) - my budujemy bardzo małą wersję tej architektury (decoder-only).

Skala: GPT-2 small ma ok. 124M parametrów i 12 warstw. Nasz model będzie miał ok. **0.2M parametrów i 4 warstwy** - czyli ~600x mniejszy. Ale architektura jest dosłownie ta sama.

### Co zmienia się względem Notebooka 4?

W Notebooku 4 zbudowaliśmy **jedną głowę uwagi**. Tutaj idziemy dalej:

1. **Multi-Head Attention** - kilka głów uwagi działających równolegle, każda może wyłapywać inne wzorce.
2. **Feed-Forward** po każdej uwadze - chwila na "przemyślenie" zebranych informacji.
3. **Połączenia rezydualne** (`x + warstwa(x)`) - bez nich głębokie sieci się nie uczą.
4. **LayerNorm** - normalizacja, która stabilizuje trening.
5. **Stos kilku takich bloków** - kolejne warstwy abstrakcji.
6. **Dropout** - regularyzacja przeciwko nauczeniu się tekstu na pamięć.

Cała architektura w jednym obrazku to tzw. **decoder-only transformer** - dokładnie ten sam typ, co GPT-1/2/3/4.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

torch.manual_seed(1337)  # tradycyjny seed Karpathy'ego
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Urządzenie: {device}")

### 1. Hiperparametry

Trzymamy je w jednym miejscu, by łatwo było eksperymentować. Domyślne wartości są dobrane tak, by trening trwał kilka minut na laptopie. Jeśli masz GPU/MPS, śmiało zwiększaj.

In [ ]:
block_size = 64       # ile znaków wstecz model widzi (tzw. context length)
batch_size = 32        # ile sekwencji w jednej paczce
n_embd = 96            # wymiar embeddingu (musi być podzielny przez n_head)
n_head = 4             # ile głów uwagi
n_layer = 4            # ile bloków transformera
dropout = 0.1

learning_rate = 3e-4
max_iters = 3000
eval_every = 100
eval_iters = 20

### 2. Dane (analogicznie do Notebooka 4)

In [ ]:
with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

data = torch.tensor([char2id[c] for c in text], dtype=torch.long)
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
test_data = data[split_idx:]

def losuj_paczke(split: str = "train") -> tuple[torch.Tensor, torch.Tensor]:
    zrodlo = train_data if split == "train" else test_data
    ix = torch.randint(0, len(zrodlo) - block_size - 1, (batch_size,))
    x = torch.stack([zrodlo[i:i+block_size] for i in ix])
    y = torch.stack([zrodlo[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Słownik: {vocab_size} znaków")
print(f"Trening: {len(train_data):,} | Test: {len(test_data):,}")

### 3. Klasa `Head` - jedna głowa uwagi (z dropoutem)

Dokładnie ta sama, co w Notebooku 4, plus dropout na wagach uwagi (regularyzacja - czasem "zerujemy" niektóre połączenia, by sieć nie polegała zbyt mocno na konkretnych ścieżkach).

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd: int, head_size: int, block_size: int, dropout: float):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wagi = wagi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wagi = F.softmax(wagi, dim=-1)
        wagi = self.dropout(wagi)
        return wagi @ v

### 4. Multi-Head Attention

**Pomysł**: zamiast jednej głowy o wymiarze `n_embd`, mamy `n_head` głów po `n_embd // n_head` każda. Wyniki sklejamy (`concat`) i przepuszczamy przez warstwę liniową (`projection`).

Dlaczego to działa lepiej? Każda głowa może uczyć się innych wzorców - jedna może patrzeć na ostatnią literę poprzedniego słowa, druga na początek wersu, trzecia na rymy itd. To jak różne "pytania", które token zadaje równolegle.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Każda głowa zwraca (B, T, head_size) - sklejamy je do (B, T, n_embd)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

### 5. Feed-Forward

Po wymianie informacji między tokenami (uwaga) każdy token z osobna "przemyśliwa" zebraną informację. To zwykły MLP: liniowa → ReLU → liniowa.

Konwencja z artykułu *Attention is all you need*: wewnętrzny wymiar to `4 * n_embd`.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

### 6. Blok transformera: uwaga + FFN + rezydualne + LayerNorm

Złożenie powyższych w jeden "blok". Dwie ważne rzeczy:

**Połączenia rezydualne** (`x = x + warstwa(x)`) - zamiast zastępować `x` wynikiem warstwy, dodajemy go. Dzięki temu gradient płynnie przepływa przez wiele warstw - bez tego głęboka sieć (np. 12-warstwowa GPT-2) by się nie nauczyła.

**LayerNorm przed warstwą** (`pre-norm`) - normalizujemy aktywacje przed wejściem do uwagi/FFN. To stabilizuje trening. Oryginalny transformer miał `post-norm` (LN po warstwie), ale w GPT-2 i nowszych standardem jest `pre-norm` - i my też tak robimy.

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = FeedForward(n_embd, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm + residual: x = x + warstwa(LN(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

### 7. Cały model: MiniGPT

Składamy wszystko w GPT:
1. **Embedding tokenów** (`vocab_size → n_embd`)
2. **Embedding pozycji** (`block_size → n_embd`) - sumujemy z tokenami
3. **Stos `n_layer` bloków transformera**
4. **Końcowy LayerNorm**
5. **Głowa językowa** (`n_embd → vocab_size`) - daje logity dla każdego znaku w słowniku

Plus metoda `generuj` z temperaturą do generowania tekstu.

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, n_embd: int, n_head: int, n_layer: int,
                 block_size: int, dropout: float):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generuj(self, idx: torch.Tensor, max_nowych: int, temperatura: float = 1.0,
                top_k: int | None = None) -> torch.Tensor:
        """Dopisuje max_nowych tokenów do idx (B, T).
        
        top_k - jeśli ustawione, bierzemy pod uwagę tylko k najbardziej prawdopodobnych
        znaków. Pomaga w jakości tekstu (mniej dziwnych liter).
        """
        for _ in range(max_nowych):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)
            logits = logits[:, -1, :] / temperatura
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Liczba parametrów: {n_params:,} (~{n_params/1e6:.2f}M)")
print(model)

### 8. Test "na sucho" przed treningiem

Zanim rozpoczniemy trening, zobaczmy, co generuje model z losowymi wagami. Powinno być to zupełnie losowe, bezsensowne. Strata powinna być w okolicach `ln(vocab_size)` ("losowo zgaduję jeden znak z `vocab_size`").

In [ ]:
import math
print(f"Strata losowa (oczekiwana): {math.log(vocab_size):.3f}")

xb, yb = losuj_paczke("train")
logits = model(xb)
B, T, V = logits.shape
loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))
print(f"Strata przed treningiem: {loss.item():.3f}")

# Generujemy z pustego kontekstu
kontekst = torch.zeros((1, 1), dtype=torch.long, device=device)
wynik = model.generuj(kontekst, max_nowych=200)[0]
print("\nLosowy bełkot (przed treningiem):")
print("".join(id2char[i.item()] for i in wynik))

### 9. Trening

Standardowa pętla z AdamW. Co `eval_every` iteracji liczymy stratę na zbiorze treningowym i testowym (uśrednioną z kilku paczek, by nie skakała).

In [ ]:
@torch.no_grad()
def oszacuj_strate() -> dict[str, float]:
    out = {}
    model.eval()
    for split in ["train", "test"]:
        straty = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = losuj_paczke(split)
            logits = model(xb)
            B, T, V = logits.shape
            straty[k] = F.cross_entropy(logits.view(B*T, V), yb.view(B*T)).item()
        out[split] = straty.mean().item()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
liveloss = PlotLosses()

for it in range(max_iters):
    xb, yb = losuj_paczke("train")
    logits = model(xb)
    B, T, V = logits.shape
    loss = F.cross_entropy(logits.view(B*T, V), yb.view(B*T))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if it % eval_every == 0 or it == max_iters - 1:
        straty = oszacuj_strate()
        liveloss.update({"loss": straty["train"], "val_loss": straty["test"]})
        liveloss.send()

print("Trening zakończony.")

### 10. Generowanie tekstu

Sprawdzamy, co nauczył się nasz mini-GPT. Możemy zaczynać od pustego kontekstu albo dowolnego ciągu z tekstu.

In [ ]:
def generuj_tekst(start_str: str = "\n", dlugosc: int = 500,
                  temperatura: float = 1.0, top_k: int | None = None) -> str:
    if start_str == "":
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)
    else:
        idx = torch.tensor([[char2id[c] for c in start_str]], dtype=torch.long, device=device)
    out = model.generuj(idx, max_nowych=dlugosc, temperatura=temperatura, top_k=top_k)
    return "".join(id2char[i.item()] for i in out[0])

print(generuj_tekst("Litwo, ojczyzno moja", dlugosc=600, temperatura=1.0))

### 11. Eksperymenty: temperatura i top-k

Ten sam model, różne strategie próbkowania:
- **Niska temperatura** (np. 0.5) - tekst bezpieczny, ale powtarzalny.
- **Wysoka temperatura** (np. 1.5) - bardziej kreatywny, ale więcej dziwactw.
- **`top_k`** - bierzemy pod uwagę tylko `k` najbardziej prawdopodobnych znaków na każdym kroku. Dobre, by uciąć ogon najgłupszych opcji.

In [ ]:
for temp in [0.5, 1.0, 1.5]:
    print(f"\n=== Temperatura = {temp} ===")
    print(generuj_tekst("Soplica", dlugosc=300, temperatura=temp))

In [ ]:
print("=== top_k = 10, temperatura = 1.0 ===")
print(generuj_tekst("\n", dlugosc=400, temperatura=1.0, top_k=10))

### 12. Co dalej, czyli skala

Zbudowaliśmy działający mini-GPT (~0.2M parametrów). Co potrzeba, by zrobić z niego "prawdziwy" model?

| Model | Parametry | Warstwy | Embedding | Kontekst |
|---|---|---|---|---|
| Nasz mini-GPT | ~0.2M | 4 | 96 | 64 |
| GPT-2 small | 124M | 12 | 768 | 1024 |
| GPT-2 large | 1.5B | 48 | 1600 | 1024 |
| GPT-3 | 175B | 96 | 12288 | 2048 |
| GPT-4 | ~1T (szac.) | ? | ? | 128k+ |

Architektura jest **dosłownie ta sama**. Różnice:
- **Skala** - więcej warstw, większy embedding, dłuższy kontekst.
- **Dane** - zamiast jednej książki: cały internet, książki, kod (TB tekstu).
- **Tokenizator** - zamiast pojedynczych znaków: BPE / tiktoken (np. ~50k tokenów-fragmentów).
- **Optymalizacje** - flash attention, mixed precision, distributed training.
- **Fine-tuning + RLHF** - by z modelu "dopisującego tekst" zrobić asystenta.

Pomysły na pobawienie się:
1. Powiększ `n_embd`, `n_layer`, `block_size` - jak zmienia się jakość?
2. Zamień tokenizator na słowa (jak w Notebooku 2) - ile parametrów teraz potrzeba?
3. Wytrenuj na innym tekście (innej książce z Wolnych Lektur).
4. Zaimplementuj BPE - prawdziwy GPT na Panu Tadeuszu.
5. Sprawdź, jakie wzorce wyłapują poszczególne głowy uwagi (wizualizacja macierzy `wagi`).